# Using Kerchunk and Icechunk with grib2io and Xarray

This notebook demonstrates how to virtualize GRIB2 files using Kerchunk and store them in a cloud-native, versioned format using Icechunk.

## Notebook Convention

Use the standard grib2io xarray backend interface directly.
Pass grib2io-specific options as keyword arguments to `xr.open_dataset(...)` or `xr.open_mfdataset(...)` (for example `use_icechunk`, `storage_options`, `filters`, `max_workers`, `network_timeout`, and `max_concurrent_requests`).

In [1]:
import grib2io
import xarray as xr
import pathlib
import os
import json
from grib2io.kerchunk import ReferenceGenerator

/opt/homebrew/Caskroom/miniforge/base/envs/grib2io/lib/python3.14/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


## 1. Generate Kerchunk References
We use `ReferenceGenerator` to scan a GRIB2 file and create a manifest of byte-range references.

In [2]:
# Use the GFS test file bundled with grib2io.
# Assumes the notebook is run from the project root or demos/ directory.
_here = pathlib.Path(os.path.abspath(""))
_candidates = [
    _here / "tests" / "input_data" / "gfs.t00z.pgrb2.1p00.f024",
    _here.parent / "tests" / "input_data" / "gfs.t00z.pgrb2.1p00.f024",
]
grib2_file = next(str(p) for p in _candidates if p.exists())

gen = ReferenceGenerator(grib2_file)
manifest = gen.generate()
print(f"Generated manifest with {len(manifest['refs'])} references")

Generated manifest with 1231 references


## 2. Open as an xarray Dataset

Use the standard grib2io/xarray interface by opening the generated manifest with
`engine="grib2io"`. This keeps dataset access routed through the existing UI.

Notebook convention: pass grib2io-specific options directly as keyword arguments
to `xr.open_dataset(...)` / `xr.open_mfdataset(...)` (rather than nesting them
under a separate backend args dict).

For an even shorter path when you have the file URL directly, use:
`xr.open_dataset(url, engine="grib2io", use_icechunk=True)`


In [3]:
manifest_path = pathlib.Path(grib2_file).with_suffix(".kerchunk.json")
manifest_path.write_text(json.dumps(manifest))

ds = xr.open_dataset(manifest_path, engine="grib2io")
display(ds)

<xarray.Dataset> Size: 195MB
Dimensions:                              (valid_time: 1, mean_sea_level: 1,
                                          y: 181, x: 360, hybrid_level: 1,
                                          hybrid_level_2: 2,
                                          entire_atmosphere: 1, surface: 1,
                                          planetary_boundary_layer: 1,
                                          isobaric_surface: 41,
                                          ...
                                          highest_tropospheric_freezing_level: 1,
                                          pressure_difference_from_ground: 1,
                                          pressure_difference_from_ground_2: 3,
                                          sigma_level: 4, sigma_level_2: 1,
                                          pressure_difference_from_ground_3: 1,
                                          potential_vorticity_surface: 2)
Coordinates: (12/49)
  * valid_time                           (valid_time) datetime64[ns] 8B 2022-...
  * mean_sea_level                       (mean_sea_level) float64 8B nan
    latitude                             (y, x) float64 521kB ...
    longitude                            (y, x) float64 521kB ...
  * hybrid_level                         (hybrid_level) float64 8B 1.0
  * hybrid_level_2                       (hybrid_level_2) float64 16B 1.0 2.0
    ...                                   ...
  * pressure_difference_from_ground      (pressure_difference_from_ground) float64 8B ...
  * pressure_difference_from_ground_2    (pressure_difference_from_ground_2) float64 24B ...
  * sigma_level                          (sigma_level) float64 32B 0.33 ... 0.72
  * sigma_level_2                        (sigma_level_2) float64 8B 0.995
  * pressure_difference_from_ground_3    (pressure_difference_from_ground_3) float64 8B ...
  * potential_vorticity_surface          (potential_vorticity_surface) float64 16B ...
Dimensions without coordinates: y, x
Data variables: (12/170)
    PRMSL                                (valid_time, mean_sea_level, y, x) float32 261kB ...
    CLMR                                 (valid_time, hybrid_level, y, x) float32 261kB ...
    ICMR                                 (valid_time, hybrid_level, y, x) float32 261kB ...
    RWMR                                 (valid_time, hybrid_level, y, x) float32 261kB ...
    SNMR                                 (valid_time, hybrid_level, y, x) float32 261kB ...
    GRLE                                 (valid_time, hybrid_level, y, x) float32 261kB ...
    ...                                   ...
    UGRD_8                               (valid_time, potential_vorticity_surface, y, x) float32 521kB ...
    VGRD_8                               (valid_time, potential_vorticity_surface, y, x) float32 521kB ...
    TMP_11                               (valid_time, potential_vorticity_surface, y, x) float32 521kB ...
    HGT_7                                (valid_time, potential_vorticity_surface, y, x) float32 521kB ...
    PRES_12                              (valid_time, potential_vorticity_surface, y, x) float32 521kB ...
    VWSH_1                               (valid_time, potential_vorticity_surface, y, x) float32 521kB ...
Attributes:
    history:  2026-06-04 17:02:59 UTC: Initialized via grib2io xarray backend...

## 4. Efficient Variable Streaming
Because the data is lazily indexed, we can "stream" or access specific parts of a variable without loading the entire dataset into memory. Using `chunks={}` in `open_dataset` ensures that Xarray uses Dask for lazy loading.

This is particularly powerful for large variables like temperature (TMP) or humidity (RH) across multiple vertical levels or time steps.

In [4]:
if "TMP" in ds.data_vars:
    # Select a spatial subset — only triggers byte-range requests for the relevant GRIB messages.
    tmp_subset = ds.TMP.isel(y=slice(0, 100), x=slice(0, 100))

    # Trigger a compute to see the data
    data = tmp_subset.compute()
    print("Successfully streamed TMP subset data.")
    display(data)

Successfully streamed TMP subset data.


<xarray.DataArray 'TMP' (valid_time: 1, isobaric_surface: 41, y: 100, x: 100)> Size: 2MB
dask.array<getitem, shape=(1, 41, 100, 100), dtype=float32, chunksize=(1, 1, 100, 100), chunktype=numpy.ndarray>
Coordinates:
  * valid_time        (valid_time) datetime64[ns] 8B 2022-11-08
  * isobaric_surface  (isobaric_surface) float64 328B 1.0 2.0 ... 9.75e+04 1e+05
    latitude          (y, x) float64 80kB dask.array<chunksize=(100, 100), meta=np.ndarray>
    longitude         (y, x) float64 80kB dask.array<chunksize=(100, 100), meta=np.ndarray>
Dimensions without coordinates: y, x
Attributes:
    discipline:                0
    parameterCategory:         0
    parameterNumber:           0
    typeOfFirstFixedSurface:   100
    valueOfFirstFixedSurface:  1.0
    valid_time:                2022-11-08T00:00:00
    shortName:                 TMP
    fullName:                  Temperature
    units:                     K

## 5. Multiple Variable Streaming
We can also stream multiple variables simultaneously. Dask will manage the parallel fetch of these chunks from the underlying storage (e.g. S3 or local disk) via the Icechunk references.

In [5]:
# Select multiple variables for a specific region
available = [v for v in ["TMP", "HGT"] if v in ds.data_vars]
if available:
    subset_ds = ds[available].isel(y=slice(50, 150), x=slice(50, 150))

    # Dask manages parallel byte-range fetches from the Icechunk store.
    computed_result = subset_ds.compute()
    print(f"Streamed {available} successfully.")
    display(computed_result)

Streamed ['TMP', 'HGT'] successfully.


<xarray.Dataset> Size: 3MB
Dimensions:           (valid_time: 1, isobaric_surface: 41, y: 100, x: 100)
Coordinates:
  * valid_time        (valid_time) datetime64[ns] 8B 2022-11-08
  * isobaric_surface  (isobaric_surface) float64 328B 1.0 2.0 ... 9.75e+04 1e+05
    latitude          (y, x) float64 80kB dask.array<chunksize=(100, 100), meta=np.ndarray>
    longitude         (y, x) float64 80kB dask.array<chunksize=(100, 100), meta=np.ndarray>
Dimensions without coordinates: y, x
Data variables:
    TMP               (valid_time, isobaric_surface, y, x) float32 2MB dask.array<chunksize=(1, 1, 100, 100), meta=np.ndarray>
    HGT               (valid_time, isobaric_surface, y, x) float32 2MB dask.array<chunksize=(1, 1, 100, 100), meta=np.ndarray>
Attributes:
    history:  2026-06-04 17:02:59 UTC: Initialized via grib2io xarray backend...